# Phase 9A — SchemaLinker Stage 1 CoT SFT

Fine-tunes **Qwen3-8B** with QLoRA (4-bit NF4) on `sql_cot_train.json` (6748 entries).
Checkpoints saved to Google Drive every ~20 min. Re-running after a disconnect resumes automatically.

| | T4 | A100 |
|---|---|---|
| VRAM | 15 GB | 40 GB |
| Config | QLoRA 4-bit, batch=2, max_len=256 | bf16, batch=8, max_len=1024 |
| Est. time | ~3–4 hours | ~45 min |

> **After stopping training mid-run:** always do **Runtime → Restart session** before restarting.
> This clears VRAM so the new run can load the model cleanly.

## Cell 1 — Verify GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
                       capture_output=True, text=True)
print(result.stdout)
gpu = result.stdout
if 'T4' in gpu:
    print("✅ T4 detected — use --gpu t4")
elif 'A100' in gpu:
    print("✅ A100 detected — use --gpu a100")
else:
    print("⚠️  Unknown GPU — check Runtime → Change runtime type")

## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

CHECKPOINT_DIR = '/content/drive/MyDrive/codegen/checkpoints/sl_sql_stage1'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoint dir: {CHECKPOINT_DIR} ✅")

## Cell 3 — Clone or update repo

In [ ]:
%%bash
set -e

REPO_DIR="/content/Codegen"
BRANCH="phase/9a-schema-linker-stage1"

if [ -d "$REPO_DIR/.git" ]; then
    echo "Repo exists — pulling latest"
    cd "$REPO_DIR"
    git fetch origin
    git checkout $BRANCH
    git pull origin $BRANCH
else
    echo "Cloning repo"
    git clone https://github.com/kethansplunk/Codegen.git "$REPO_DIR"
    cd "$REPO_DIR"
    git checkout $BRANCH
fi

echo "Branch : $(git branch --show-current)"
echo "Commit : $(git log --oneline -1)"
echo "Files  : $(ls Data/cot_data/)"


## Cell 4 — Install dependencies

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes FlagEmbedding
print("Dependencies installed ✅")

## Cell 5 — Verify training data and VRAM

In [ ]:
import json, os, subprocess

# Check training data
DATA_PATH = '/content/Codegen/Data/cot_data/sql_cot_train.json'
with open(DATA_PATH) as f:
    data = json.load(f)
print(f"Training samples : {len(data)}")
print(f"Keys             : {list(data[0].keys())}")
assert len(data) >= 6000, "Expected at least 6000 samples"
print("Data check ✅")

# Check free VRAM
result = subprocess.run(['nvidia-smi', '--query-gpu=memory.used,memory.free,memory.total',
                         '--format=csv,noheader'], capture_output=True, text=True)
print(f"\nVRAM: {result.stdout.strip()}")

# Check existing checkpoints on Drive
ckpt_dir = '/content/drive/MyDrive/codegen/checkpoints/sl_sql_stage1'
ckpts = sorted([d for d in os.listdir(ckpt_dir) if d.startswith('checkpoint-')])
if ckpts:
    print(f"\nExisting checkpoints: {ckpts}")
    print(f"Will resume from    : {ckpts[-1]}")
else:
    print("\nNo checkpoints found — will train from scratch")

## Cell 6 — Run training

- **First run:** trains from scratch, checkpoints every ~20 min to Drive
- **After disconnect:** re-run this cell — auto-resumes from latest Drive checkpoint
- **After stopping mid-run:** do Runtime → Restart session first, then re-run Cells 2–4 and this cell
- Change `--gpu a100` if you switched to A100 runtime

In [ ]:
%%bash
cd /content/Codegen

# Disable tqdm — progress printed by StatusCallback instead
export TQDM_DISABLE=1

python -m src.schema_linker.train_stage1 \
    --data  Data/cot_data/sql_cot_train.json \
    --model Qwen/Qwen3-8B \
    --out   /content/drive/MyDrive/codegen/checkpoints/sl_sql_stage1 \
    --gpu   t4
    # change --gpu t4 to --gpu a100 if on A100 runtime


## Cell 7 — Check training status (run anytime while Cell 6 is active)

In [ ]:
import json, os

ckpt_dir = '/content/drive/MyDrive/codegen/checkpoints/sl_sql_stage1'

# List all checkpoints
ckpts = sorted([d for d in os.listdir(ckpt_dir) if d.startswith('checkpoint-')])
print(f"Checkpoints saved: {ckpts}")

# Read trainer state
state_file = os.path.join(ckpt_dir, 'trainer_state.json')
if os.path.exists(state_file):
    with open(state_file) as f:
        state = json.load(f)
    step    = state['global_step']
    total   = 1140
    pct     = step / total * 100
    best    = state.get('best_metric', 'N/A')
    last    = state['log_history'][-1] if state['log_history'] else {}
    print(f"\nStep     : {step} / {total}  ({pct:.1f}%)")
    print(f"Best loss: {best}")
    print(f"Last log : {last}")
else:
    print("No trainer_state.json yet — checkpoint hasn't saved")

## Cell 8 — Verify final adapter output (run after training completes)

In [ ]:
import os

CHECKPOINT_DIR = '/content/drive/MyDrive/codegen/checkpoints/sl_sql_stage1'
print("Files in output dir:")
for f in sorted(os.listdir(CHECKPOINT_DIR)):
    path = os.path.join(CHECKPOINT_DIR, f)
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f"  {f:<50} {size_mb:>8.1f} MB")
    else:
        print(f"  {f}/")

adapter = os.path.join(CHECKPOINT_DIR, 'adapter_model.safetensors')
if os.path.exists(adapter):
    size = os.path.getsize(adapter) / 1e6
    print(f"\nAdapter size: {size:.0f} MB  (expected ~260 MB) ✅")
else:
    print("\nadapter_model.safetensors not found — training may still be running.")